# Experiment 03 — Simple CNN on Focal Clips

**New approach:** Train a lightweight 4-layer CNN on mel spectrograms of the 35k focal clips (30% subset). This gives a completely independent signal from Perch — trained on labelled individual-bird recordings rather than derived from a pre-trained embedding model.

**Hypothesis:** Even a simple CNN trained on 10k focal clips will capture useful discriminative features that are complementary to Perch embeddings. Ensembling the two should improve overall cmAP.

**Strategy to handle CPU speed:** Pre-compute and cache mel spectrograms once as float16 .npy files so each training epoch only does fast disk-reads + neural-net forward/backward.

In [1]:
import ast, gc, warnings, time
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
import onnxruntime as ort
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.special import expit as sigmoid
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

DATA_DIR        = Path("../data")
AUDIO_DIR       = DATA_DIR / "train_audio"
SOUNDSCAPE_DIR  = DATA_DIR / "train_soundscapes"
PERCH_ONNX      = Path("../data/models/perch/perch_v2_no_dft.onnx")
PERCH_LABELS    = Path("../data/models/perch_tf/assets/labels.csv")
CACHE_DIR       = Path("../data/cache")
MEL_CACHE_DIR   = CACHE_DIR / "exp03_mels"
CACHE_DIR.mkdir(exist_ok=True)
MEL_CACHE_DIR.mkdir(exist_ok=True)

SR          = 32_000
WIN_SAMPLES = SR * 5
N_FFT       = 1024
HOP         = 512
N_MELS      = 64
N_CLASSES   = 234
BATCH_SIZE  = 32
EPOCHS      = 5
LR          = 1e-3
SUBSET_FRAC = 0.30

device = torch.device("cpu")
print(f"Device: {device} | torch {torch.__version__}")

Device: cpu | torch 2.2.2


In [2]:
# ── Taxonomy mapping ─────────────────────────────────────────────────────────
taxonomy     = pd.read_csv(DATA_DIR / "taxonomy.csv")
taxon_ids    = taxonomy["primary_label"].astype(str).tolist()
taxon_to_idx = {tid: i for i, tid in enumerate(taxon_ids)}

perch_labels = pd.read_csv(PERCH_LABELS, header=0)
perch_labels.columns = ["scientific_name"]
perch_labels["bc_index"] = np.arange(len(perch_labels))
mapping = taxonomy.merge(perch_labels, on="scientific_name", how="left")
comp_to_bc = {}
for _, row in mapping[mapping["bc_index"].notna()].iterrows():
    comp_to_bc[taxon_to_idx[str(row["primary_label"])]] = int(row["bc_index"])
print(f"{N_CLASSES} classes | {len(comp_to_bc)} mapped to Perch")

234 classes | 203 mapped to Perch


In [3]:
# ── 30% stratified subset of focal clips ────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")

# groupby().sample() is safe across pandas versions
subset = (
    train_df
    .groupby("primary_label", group_keys=False)
    .sample(frac=SUBSET_FRAC, random_state=42)
    .reset_index(drop=True)
)

# Only keep clips where the species exists in taxonomy
subset = subset[subset["primary_label"].astype(str).isin(taxon_to_idx)].reset_index(drop=True)

print(f"Full dataset: {len(train_df)} | 30% subset: {len(subset)} clips, {subset['primary_label'].nunique()} species")

Full dataset: 35549 | 30% subset: 10666 clips, 202 species


In [4]:
# ── Mel spectrogram transform ────────────────────────────────────────────────
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP,
    n_mels=N_MELS, f_min=20.0, f_max=16000.0,
)
db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)

def audio_to_mel(path, target_samples=WIN_SAMPLES):
    """Load audio file, pad/crop to target_samples, return mel spec as (1, N_MELS, T)."""
    try:
        y, sr = sf.read(str(path), dtype="float32", always_2d=False)
    except Exception:
        return np.zeros((1, N_MELS, target_samples // HOP + 1), dtype=np.float16)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if len(y) < target_samples:
        y = np.pad(y, (0, target_samples - len(y)))
    else:
        # Random crop for training (first call during pre-compute uses start=0)
        start = np.random.randint(0, len(y) - target_samples + 1) if len(y) > target_samples else 0
        y = y[start : start + target_samples]
    wav = torch.from_numpy(y).unsqueeze(0)  # (1, T)
    mel = db_transform(mel_transform(wav))  # (1, n_mels, frames)
    # Normalize per-instance
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.numpy().astype(np.float16)

# Quick shape test
test_shape = audio_to_mel(AUDIO_DIR / subset.iloc[0]["filename"]).shape
print(f"Mel shape: {test_shape}  (channels, mels, time-frames)")
T_FRAMES = test_shape[2]

Mel shape: (1, 64, 313)  (channels, mels, time-frames)


In [5]:
# ── Pre-compute and cache mel spectrograms ───────────────────────────────────
# One-time cost; subsequent runs load from disk (fast).
def mel_cache_path(filename):
    safe = filename.replace("/", "__")
    return MEL_CACHE_DIR / (safe + ".npy")

missing = [row for _, row in subset.iterrows() if not mel_cache_path(row["filename"]).exists()]
print(f"Cached: {len(subset)-len(missing)}/{len(subset)} | To compute: {len(missing)}")

if missing:
    t0 = time.time()
    for row in tqdm(missing, desc="Pre-computing mels"):
        mel = audio_to_mel(AUDIO_DIR / row["filename"])
        np.save(mel_cache_path(row["filename"]), mel)
    print(f"Done in {time.time()-t0:.0f}s")
else:
    print("All mels already cached.")

Cached: 0/10666 | To compute: 10666


Pre-computing mels:   0%|          | 0/10666 [00:00<?, ?it/s]

Done in 421s


In [6]:
# ── Dataset + DataLoader ─────────────────────────────────────────────────────
class FocalDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row  = self.df.iloc[i]
        path = mel_cache_path(row["filename"])
        mel  = np.load(path).astype(np.float32)  # (1, n_mels, T)

        # SpecAugment: random frequency / time masking
        if self.augment:
            mel = mel.copy()
            # freq mask (up to 8 bins)
            f0 = np.random.randint(0, N_MELS - 8)
            mel[:, f0:f0+np.random.randint(1,8), :] = 0
            # time mask (up to 20 frames)
            t0 = np.random.randint(0, T_FRAMES - 20)
            mel[:, :, t0:t0+np.random.randint(1,20)] = 0

        label = np.zeros(N_CLASSES, dtype=np.float32)
        pid = str(row["primary_label"])
        if pid in taxon_to_idx:
            label[taxon_to_idx[pid]] = 1.0

        return torch.from_numpy(mel), torch.from_numpy(label)

train_ds = FocalDataset(subset, augment=True)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print(f"Dataset: {len(train_ds)} clips | {len(train_dl)} batches/epoch")

Dataset: 10666 clips | 334 batches/epoch


In [7]:
# ── Model definition ─────────────────────────────────────────────────────────
class MelCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, n_classes))

    def forward(self, x):
        return self.head(self.features(x))

model = MelCNN(N_CLASSES).to(device)
params = sum(p.numel() for p in model.parameters())
print(f"MelCNN: {params/1e6:.2f}M params")

MelCNN: 0.45M params


In [8]:
# ── Training loop ────────────────────────────────────────────────────────────
MODEL_CKPT = CACHE_DIR / "exp03_melcnn.pt"

if MODEL_CKPT.exists():
    print("Loading saved checkpoint...")
    model.load_state_dict(torch.load(MODEL_CKPT, map_location=device))
else:
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses, t0 = [], time.time()
        for mel, label in tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
            optimizer.zero_grad()
            loss = criterion(model(mel.to(device)), label.to(device))
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        scheduler.step()
        print(f"  Epoch {epoch}: loss={np.mean(losses):.4f}  [{time.time()-t0:.0f}s]")

    torch.save(model.state_dict(), MODEL_CKPT)
    print("Checkpoint saved.")

Epoch 1/5:   0%|          | 0/334 [00:00<?, ?it/s]

  Epoch 1: loss=0.0497  [265s]


Epoch 2/5:   0%|          | 0/334 [00:00<?, ?it/s]

  Epoch 2: loss=0.0257  [230s]


Epoch 3/5:   0%|          | 0/334 [00:00<?, ?it/s]

  Epoch 3: loss=0.0253  [270s]


Epoch 4/5:   0%|          | 0/334 [00:00<?, ?it/s]

  Epoch 4: loss=0.0250  [248s]


Epoch 5/5:   0%|          | 0/334 [00:00<?, ?it/s]

  Epoch 5: loss=0.0247  [267s]
Checkpoint saved.


In [9]:
# ── Evaluate CNN on labeled soundscape windows ───────────────────────────────
def time_to_sec(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

raw_df    = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
raw_df["start_sec"] = raw_df["start"].apply(time_to_sec)
labels_df = (
    raw_df.drop_duplicates(subset=["filename", "start_sec"])
    .sort_values(["filename", "start_sec"]).reset_index(drop=True)
)

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(";"):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

Y = np.stack(labels_df["primary_label"].apply(parse_labels).values)

# Run CNN on each soundscape window
model.eval()
cnn_preds = []

with torch.no_grad():
    for fname, grp in tqdm(labels_df.groupby("filename"), desc="Soundscape files"):
        audio, _ = sf.read(str(SOUNDSCAPE_DIR / fname), dtype="float32", always_2d=False)
        batch_mels = []
        for _, row in grp.iterrows():
            start = int(row["start_sec"]) * SR
            chunk = audio[start : start + WIN_SAMPLES]
            if len(chunk) < WIN_SAMPLES:
                chunk = np.pad(chunk, (0, WIN_SAMPLES - len(chunk)))
            wav = torch.from_numpy(chunk).unsqueeze(0)
            mel = db_transform(mel_transform(wav))
            mel = (mel - mel.mean()) / (mel.std() + 1e-6)
            batch_mels.append(mel.numpy().astype(np.float32))

        batch  = torch.from_numpy(np.stack(batch_mels))
        logits = model(batch)
        cnn_preds.append(torch.sigmoid(logits).numpy())

cnn_preds = np.concatenate(cnn_preds, axis=0)
print(f"CNN predictions: {cnn_preds.shape}")

Soundscape files:   0%|          | 0/66 [00:00<?, ?it/s]

CNN predictions: (739, 234)


In [10]:
# ── Load Perch LR-probe predictions from exp02 cache ────────────────────────
EMB_CACHE = CACHE_DIR / "exp02_embeddings.npy"
LOG_CACHE = CACHE_DIR / "exp02_logits.npy"
emb_full    = np.load(EMB_CACHE)
logits_full = np.load(LOG_CACHE)

scores_raw = np.zeros((len(labels_df), N_CLASSES), dtype=np.float32)
for comp_idx, bc_idx in comp_to_bc.items():
    scores_raw[:, comp_idx] = logits_full[:, bc_idx]

groups = labels_df["filename"].values
gkf    = GroupKFold(n_splits=5)
perch_oof = sigmoid(scores_raw).copy()

for fold, (tr, va) in enumerate(gkf.split(emb_full, groups=groups)):
    scaler = StandardScaler()
    pca    = PCA(n_components=64, whiten=True, random_state=42)
    emb_tr = pca.fit_transform(scaler.fit_transform(emb_full[tr]))
    emb_va = pca.transform(scaler.transform(emb_full[va]))
    X_tr = np.concatenate([emb_tr, scores_raw[tr]], axis=1)
    X_va = np.concatenate([emb_va, scores_raw[va]], axis=1)
    active = [c for c in range(N_CLASSES) if Y[tr, c].sum() >= 3]
    for c in active:
        clf = LogisticRegression(C=0.5, max_iter=1000, solver="lbfgs")
        clf.fit(X_tr, Y[tr, c])
        perch_oof[va, c] = clf.predict_proba(X_va)[:, 1]

print("Perch LR-probe OOF ready")

Perch LR-probe OOF ready


In [11]:
# ── Metrics + comparison ─────────────────────────────────────────────────────
def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        yt = np.concatenate([y_true[:, c], np.ones(pad)])
        yp = np.concatenate([y_pred[:, c], np.ones(pad)])
        aps.append(average_precision_score(yt, yp))
    return np.mean(aps)

def macro_auc(y_true, y_pred):
    aucs = []
    for c in range(y_true.shape[1]):
        if 0 < y_true[:, c].sum() < len(y_true):
            aucs.append(roc_auc_score(y_true[:, c], y_pred[:, c]))
    return (np.mean(aucs) if aucs else 0.0), len(aucs)

# Ensemble: blend CNN + Perch
for w in [0.3, 0.5, 0.7]:
    blend = w * cnn_preds + (1-w) * perch_oof
    a, n  = macro_auc(Y, blend)
    c     = padded_cmap(Y, blend)
    print(f"  CNN×{w:.1f} + Perch×{1-w:.1f}  →  AUC: {a:.4f} ({n} cls) | cmAP: {c:.4f}")

print()
print("=" * 62)
print("Experiment 03 Summary")
print("=" * 62)
print(f"{'Method':<35} {'Macro AUC':>10} {'Padded cmAP':>12}")
print("-" * 62)

a, n = macro_auc(Y, cnn_preds)
c    = padded_cmap(Y, cnn_preds)
print(f"{'CNN only (30% focal clips)':<35} {a:>10.4f} {c:>12.4f}")

a, n = macro_auc(Y, perch_oof)
c    = padded_cmap(Y, perch_oof)
print(f"{'Perch LR probes (exp02)':<35} {a:>10.4f} {c:>12.4f}")

best_w = 0.3
blend  = best_w * cnn_preds + (1-best_w) * perch_oof
a, n   = macro_auc(Y, blend)
c      = padded_cmap(Y, blend)
print(f"{'Ensemble (CNN×0.3 + Perch×0.7)':<35} {a:>10.4f} {c:>12.4f}")
print("=" * 62)

  CNN×0.3 + Perch×0.7  →  AUC: 0.8848 (75 cls) | cmAP: 0.8953


  CNN×0.5 + Perch×0.5  →  AUC: 0.8825 (75 cls) | cmAP: 0.8947


  CNN×0.7 + Perch×0.3  →  AUC: 0.8782 (75 cls) | cmAP: 0.8935

Experiment 03 Summary
Method                               Macro AUC  Padded cmAP
--------------------------------------------------------------


CNN only (30% focal clips)              0.4866       0.8071


Perch LR probes (exp02)                 0.8891       0.8955


Ensemble (CNN×0.3 + Perch×0.7)          0.8848       0.8953
